# Contrastive Siamese Network for Sensor-Based Gesture Recognition

This notebook implements a **Supervised Contrastive Siamese Network** for the Kaggle challenge. 
It utilizes the temporal `SequenceExtractor` from `base_utils_qwen` to generate rich multi-domain features.

**Architecture Options:**
- **Backbone:** 1D CNN with dilated/standard convolutions.
- **Temporal Aggregation:** Bidirectional GRU (`gru`), Transformer Encoder (`attention`), or Global Pooling (`pool`).
- **Sequence Masking:** Automatically masks padded timesteps so temporal models don't process noise.

We provide both `GridSearchCV` and `BayesSearchCV` pipelines.

In [1]:
import sys
import os
import warnings
import itertools
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from sklearn.pipeline import Pipeline
import random
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit, train_test_split

try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-optimize", "-q"])
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

try:
    dataset_name = os.listdir("/kaggle/input/datasets/keithmarange")[0]
    sys.path.append(f"/kaggle/input/datasets/keithmarange/{dataset_name}/")
    sys.path.append("/kaggle/input/cmi-competition-code")
except Exception:
    pass

# ============================================================
# SETUP PATHS – MUST BE FIRST CELL
# ============================================================

import sys
import os

# Figure out where we are
current_dir = os.getcwd()
workspace_root = current_dir

# If we're in notebooks/ subfolder, go up one level
if os.path.basename(current_dir) == "notebooks":
    workspace_root = os.path.dirname(current_dir)

# Add paths so Python can find src/
src_path = os.path.join(workspace_root, "src")
sys.path.insert(0, workspace_root)   # So 'import src' works
sys.path.insert(0, src_path)         # So 'import data_utils' works directly

# ============================================================
# IMPORTS
# ============================================================

# Try importing with src/ prefix (local)
try:
    from src import data_utils
    from src.base_utils_qwen import (
        SequenceExtractor,
        competition_scorer,
        evaluate_holdout,
        make_competition_scorer,
        prepare_bayesian_space,
    )
    from src.utils_siamese_contrastive import KerasContrastiveSiameseClassifier
    print("✅ Imports loaded from src/")
    
except ImportError:
    # Fallback: flattened imports (Kaggle or if src not found)
    try:
        import data_utils
        from base_utils_qwen import (
            SequenceExtractor,
            competition_scorer,
            evaluate_holdout,
            make_competition_scorer,
            prepare_bayesian_space,
        )
        from utils_siamese_contrastive import KerasContrastiveSiameseClassifier
        print("✅ Imports loaded flattened")
    except ImportError as e:
        print(f"❌ Import failed: {e}")
        print(f"Current directory: {os.getcwd()}")
        print(f"Files in current directory: {os.listdir('.')}")
        print(f"Files in src/ (if exists): {os.listdir('src') if os.path.exists('src') else 'src/ not found'}")
        raise

from utils_siamese_contrastive import KerasContrastiveSiameseClassifier
from sklearn.metrics import classification_report, f1_score, make_scorer
from skopt.space import Categorical, Integer, Real
from base_utils_qwen import prepare_bayesian_space
import random
from datetime import datetime

E0000 00:00:1782696914.464920      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782696914.510882      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782696914.874866      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782696914.874903      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782696914.874905      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782696914.874907      23 computation_placer.cc:177] computation placer already registered. Please check linka

✅ Imports loaded flattened


In [2]:
# ============================================================
# CONFIGURATION
# ============================================================
target_col = "bfrb"
orientation_col = "orientation"
search_mode = "bayesian"  # "grid" or "bayesian"

random_state = 42
n_splits = 3
n_iter = 50
train_size = 0.4
error_score_constant = 0.0
verbose = 3
do_cross_val = False

# Fast local smoke test on data/eg.csv (set False for full train.csv)
use_eg_sample = False
eg_sample_pct = 0.02

# Optional row-level filters before split
do_handness = False
filter_non_bfrb_classes = False
filter_orientation_class_list = None  # e.g. ["Seated Straight"]
slice_by_orientation = None
slice_by_bfrb = None

if do_cross_val:
    cv_object = GroupKFold(n_splits=n_splits)
else:
    cv_object = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

if target_col == "bfrb":
    scorer = competition_scorer
else:
    scorer = make_scorer(f1_score, average="macro", zero_division=0)

# Smaller training budget when using the eg.csv sample
default_epochs = 8 if use_eg_sample else 30
default_patience = 3 if use_eg_sample else 8
default_batch_size = 16 if use_eg_sample else 32

print(f"Search mode: {search_mode}")
print(f"use_eg_sample: {use_eg_sample}")

Search mode: bayesian
use_eg_sample: False


In [3]:
# ============================================================
# DATA LOADING & PREPROCESSING
# ============================================================
data_root = data_utils.find_data_root()
eg_path = data_root / "eg.csv"

if use_eg_sample and eg_path.exists():
    raw_train_df = pd.read_csv(eg_path)
    train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
    print(f"Using eg.csv sample: {raw_train_df['sequence_id'].nunique()} sequences")
else:
    raw_train_df = pd.read_csv(data_root / "train.csv")
    train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
    if use_eg_sample:
        seq_ids = raw_train_df["sequence_id"].drop_duplicates().sample(
            frac=eg_sample_pct, random_state=random_state
        )
        raw_train_df = raw_train_df[raw_train_df["sequence_id"].isin(seq_ids)].copy()
        print(f"Using {eg_sample_pct:.0%} sequence sample: {raw_train_df['sequence_id'].nunique()} sequences")

train_df = raw_train_df.set_index("row_id").copy(deep=True)

if do_handness and "handedness" in train_demo_df.columns:
    train_df["handedness"] = train_df["subject"].map(train_demo_df.set_index("subject")["handedness"])
    left_handed_mask = train_df["handedness"].eq(0)
    train_df.loc[left_handed_mask, "acc_x"] *= -1.0
    train_df = train_df.drop(columns=["handedness"])

upside_down_mask = train_df["subject"].isin(["SUBJ_019262", "SUBJ_045235"])
train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0
train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0

train_df["is_target"] = (train_df["sequence_type"] == "Target").astype(bool)
train_df["bfrb"] = train_df["gesture"].where(train_df["is_target"], "non_bfrb")
train_df["gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df["gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]

if filter_non_bfrb_classes:
    train_df = train_df.loc[train_df["is_target"]].copy()

if filter_orientation_class_list is not None:
    train_df = train_df[train_df[orientation_col].isin(filter_orientation_class_list)].copy()

if slice_by_bfrb is True:
    train_df = train_df.loc[train_df["is_target"]].copy()
elif slice_by_bfrb is False:
    train_df = train_df.loc[~train_df["is_target"]].copy()

if slice_by_orientation is not None:
    train_df = train_df[train_df[orientation_col].isin(slice_by_orientation)].copy()

if filter_orientation_class_list is not None:
    sequences = train_df[["sequence_id", "is_target", target_col, orientation_col]].drop_duplicates()
    train_seqs, test_seqs = train_test_split(
        sequences["sequence_id"],
        test_size=(1 - train_size),
        stratify=sequences[target_col],
        random_state=random_state,
    )
    train_sample_df = train_df[train_df["sequence_id"].isin(train_seqs)].copy()
    hold_out_df = train_df[train_df["sequence_id"].isin(test_seqs)].copy()
else:
    try:
        train_sample_df, hold_out_df = data_utils.sample_balanced_split(
            train_df,
            train_pct=train_size,
            test_pct=min(0.2, 1 - train_size),
            random_state=random_state,
        )
    except ValueError as exc:
        print(f"Balanced split unavailable ({exc}); using simple sequence split.")
        unique_seqs = (
            train_df[["sequence_id", target_col]]
            .drop_duplicates("sequence_id")
            .sample(frac=1, random_state=random_state)
        )
        n_train = max(1, int(len(unique_seqs) * train_size))
        train_seqs = unique_seqs.iloc[:n_train]["sequence_id"]
        test_seqs = unique_seqs.iloc[n_train:]["sequence_id"]
        if len(test_seqs) == 0 and len(unique_seqs) > 1:
            test_seqs = unique_seqs.iloc[-1:]["sequence_id"]
            train_seqs = unique_seqs.iloc[:-1]["sequence_id"]
        train_sample_df = train_df[train_df["sequence_id"].isin(train_seqs)].copy()
        hold_out_df = train_df[train_df["sequence_id"].isin(test_seqs)].copy()
        print(
            f"Train: {train_sample_df['sequence_id'].nunique()} seqs | "
            f"Test: {hold_out_df['sequence_id'].nunique()} seqs"
        )

X_train = train_sample_df.copy()
X_test = hold_out_df.copy()
y_train = train_sample_df[["sequence_id", "is_target", target_col]].copy()
y_test = hold_out_df[["sequence_id", "is_target", target_col]].copy()
groups = X_train["sequence_id"]

print(f"Train sequences: {X_train['sequence_id'].nunique()} | Test sequences: {X_test['sequence_id'].nunique()}")
print(f"X_train rows: {len(X_train):,} | y_train rows: {len(y_train):,}")

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data
Train: 2910 seqs | 35.7%
Test:  969 seqs  | 11.9%
Train sequences: 2910 | Test sequences: 969
X_train rows: 213,038 | y_train rows: 213,038


In [4]:
# ============================================================
# GRID SEARCH PARAMETER SPACE (One Example Each)
# ============================================================

extractor_space_grid = {
    "extractor__acc_modes": ["smoothed|velocity|jerk"],
    "extractor__rotation_modes": ["quaternion|angular_velocity"],
    "extractor__tof_modes": ['sensor_stats'],
    "extractor__thm_modes": ["centered_diff"],
    "extractor__window_size": [5],
    "extractor__clip_value": [None],
    "extractor__interp_mode": ["linear"],
    "extractor__motion_filter_mode": [None],
    "extractor__kalman_process_noise": [1e-3],
    "extractor__kalman_measurement_noise": [1e-1],
    "extractor__padding_value": [0.0],
    # NEW parameters (add one example each)
    "extractor__native_sampling_rate": [100],
    "extractor__target_sampling_rate": [100],
    "extractor__chunk_window_size": [30],
    "extractor__chunk_stride": [10],
    "extractor__use_dead_reckoning": [False],
    "extractor__dead_reckoning_detrend": [False],
}

classifier_space_grid = {
    "classifier__target": [target_col],
    "classifier__padding_value": [0.0],
    "classifier__backbone_filters": ["128-256-256-256","256-512-512-512"],
    "classifier__kernel_sizes": ["3-3-3-3"],
    "classifier__temporal_mode": ["attention"],
    "classifier__embedding_dim": [32],
    "classifier__contrastive_weight": [0.1],
    "classifier__temperature": [0.1],
    "classifier__dense_units": ["64-32","128-64"],
    "classifier__dropout": [0.0],
    "classifier__learning_rate": [5e-4, 1e-3],
    "classifier__batch_size": [128, 256],
    "classifier__epochs": [100],
    "classifier__patience": [10],
    "classifier__verbose": [1],
    "classifier__random_state": [42],
}

# ============================================================
# BAYESIAN PARAMETER SPACE (Full Exploration)
# ============================================================

if search_mode == "bayesian" and SKOPT_AVAILABLE:
    param_space = {
        # ---------- EXTRACTOR ----------
        # Sensor modes (keep fixed to best found, but can change if needed)
        "extractor__acc_modes": Categorical(['smoothed|velocity|jerk']),
        "extractor__rotation_modes": Categorical(['quaternion|angular_velocity']),
        "extractor__tof_modes": Categorical(['sensor_stats']),
        "extractor__thm_modes": Categorical(['centered']),
        # Preprocessing
        "extractor__window_size": Categorical([5, 20, 50]),
        "extractor__clip_value": Categorical([None]),
        "extractor__interp_mode": Categorical(["linear"]),
        "extractor__motion_filter_mode": Categorical([None]),
        "extractor__kalman_process_noise": Categorical([1e-2]),
        "extractor__kalman_measurement_noise": Categorical([1e-2]),
        "extractor__padding_value": Categorical([0.0]),
        # NEW parameters
        "extractor__native_sampling_rate": Categorical([100]),  # usually fixed to raw rate
        "extractor__target_sampling_rate": Categorical([100]),
        "extractor__chunk_window_size": Categorical([10, 50, 100, 150, 300, 350]),
        "extractor__chunk_stride": Categorical([10, 50, 100, 150, 200, 300, 350]),
        "extractor__use_dead_reckoning": Categorical([False]),
        "extractor__dead_reckoning_detrend": Categorical([False]),
        
        # ---------- CLASSIFIER ----------
        "classifier__target": Categorical([target_col]),        # should match extractor maxlen
        "classifier__padding_value": Categorical([0.0]),
        "classifier__backbone_filters": Categorical([ "64-128-128-128", "128-64", "128-128-128-64"]),
        "classifier__kernel_sizes": Categorical(["3-3-3-3", "5-5-5-5"]),
        "classifier__temporal_mode": Categorical(['attention']),
        "classifier__embedding_dim":  Categorical([256]),
        "classifier__contrastive_weight": Real(0.01, 1.0, prior="uniform"),
        "classifier__temperature": Real(0.05, 0.9, prior="uniform"),
        "classifier__dense_units": Categorical(["256-128", "128-64", "512-128"]),
        "classifier__dropout": Categorical([0.0]),
        "classifier__learning_rate": Categorical([1e-4]),   # fixed to best found
        "classifier__batch_size": Categorical([128, 150, 256]),
        "classifier__epochs": Categorical([100]),
        "classifier__patience": Categorical([1000]),
        "classifier__verbose": Categorical([0]),
        "classifier__random_state": Categorical([42]),
    }
    
    # Convert complex objects to JSON strings for skopt
    param_space = prepare_bayesian_space(param_space)
    
else:
    # GRID SEARCH MODE - One example each
    param_space = {**extractor_space_grid, **classifier_space_grid}

In [5]:

# ============================================================
# PRINT SUMMARY
# ============================================================

print("=" * 60)
print("PARAMETER SPACE SUMMARY - SIAMESE NETWORK")
print("=" * 60)
print(f"Search Mode: {search_mode}")
print(f"Target Column: {target_col}")

if search_mode == "bayesian":
    print("Bayesian mode: Continuous parameter sampling")
    print(f"Total extractor parameters: {len(extractor_space_grid)}")
    print(f"Total classifier parameters: {len(classifier_space_grid)}")
    print("\n--- Feature Extractor Options (Bayesian) ---")
    print(f"  ACC_MODES: fixed to 'smoothed|velocity|jerk'")
    print(f"  ROTATION_MODES: fixed to 'quaternion|angular_velocity'")
    print(f"  TOF_MODES: fixed to 'sensor_stats'")
    print(f"  THM_MODES: fixed to 'centered'")
    print(f"  Window size: {param_space['extractor__window_size']}")
    print(f"  Motion filter: {param_space['extractor__motion_filter_mode']}")
    print(f"  Target sampling rate: {param_space['extractor__target_sampling_rate']}")
    print(f"  Chunk window size: {param_space['extractor__chunk_window_size']}")
    print(f"  Chunk stride: {param_space['extractor__chunk_stride']}")
    print(f"  Use dead reckoning: {param_space['extractor__use_dead_reckoning']}")
    print("\n--- Siamese Network Options (Bayesian) ---")
    print(f"  backbone_filters: {param_space['classifier__backbone_filters']}")
    print(f"  kernel_sizes: {param_space['classifier__kernel_sizes']}")
    print(f"  temporal_mode: {param_space['classifier__temporal_mode']}")
    print(f"  embedding_dim: {param_space['classifier__embedding_dim']}")
    print(f"  contrastive_weight: {param_space['classifier__contrastive_weight']}")
    print(f"  temperature: {param_space['classifier__temperature']}")
    print(f"  dropout: {param_space['classifier__dropout']}")
    print(f"  learning_rate: {param_space['classifier__learning_rate']} (fixed)")
    print(f"  batch_size: {param_space['classifier__batch_size']}")
else:
    print("Grid mode: Fixed single configuration")
    print(f"Total extractor parameters: {len(extractor_space_grid)}")
    print(f"Total classifier parameters: {len(classifier_space_grid)}")
    total_combos = 1
    for key, values in param_space.items():
        total_combos *= len(values)
    print(f"Total grid combinations: {total_combos:,}")
    print("\n--- Grid Parameters (One Example Each) ---")
    for key, value in param_space.items():
        print(f"  {key}: {value}")

print("\n✅ Parameter space ready for optimization.")

PARAMETER SPACE SUMMARY - SIAMESE NETWORK
Search Mode: bayesian
Target Column: bfrb
Bayesian mode: Continuous parameter sampling
Total extractor parameters: 17
Total classifier parameters: 16

--- Feature Extractor Options (Bayesian) ---
  ACC_MODES: fixed to 'smoothed|velocity|jerk'
  ROTATION_MODES: fixed to 'quaternion|angular_velocity'
  TOF_MODES: fixed to 'sensor_stats'
  THM_MODES: fixed to 'centered'
  Window size: Categorical(categories=(5, 20, 50), prior=None)
  Motion filter: Categorical(categories=(None,), prior=None)
  Target sampling rate: Categorical(categories=(100,), prior=None)
  Chunk window size: Categorical(categories=(10, 50, 100, 150, 300, 350), prior=None)
  Chunk stride: Categorical(categories=(10, 50, 100, 150, 200, 300, 350), prior=None)
  Use dead reckoning: Categorical(categories=(False,), prior=None)

--- Siamese Network Options (Bayesian) ---
  backbone_filters: Categorical(categories=('64-128-128-128', '128-64', '128-128-128-64'), prior=None)
  kernel_si

In [6]:
# ============================================================
# EXTRACTOR CHANNEL PREVIEW
# ============================================================
sample_df = X_train.head(min(2000, len(X_train)))
extractor_keys = [k for k in param_space.keys() if k.startswith("extractor__")]
is_bayesian = search_mode == "bayesian" and SKOPT_AVAILABLE

if is_bayesian:

    print("Bayesian mode: sampling 3 extractor configs for channel preview\n")
    preview_combos = []
    for _ in range(3):
        params = {}
        for key in extractor_keys:
            param_name = key.replace("extractor__", "")
            space = param_space[key]
            if isinstance(space, Categorical):
                value = random.choice(space.categories)
            elif isinstance(space, Integer):
                value = random.randint(space.low, space.high)
            elif isinstance(space, Real):
                value = random.uniform(space.low, space.high)
            else:
                value = space
            params[param_name] = value
        params["padding_value"] = 0.0
        preview_combos.append(params)
else:
    extractor_values = [param_space[k] for k in extractor_keys]
    preview_combos = [
        {k.replace("extractor__", ""): v for k, v in zip(extractor_keys, combo)}
        for combo in itertools.islice(itertools.product(*extractor_values), 3)
    ]
    for params in preview_combos:
        params["padding_value"] = 0.0

for i, extractor_params in enumerate(preview_combos, start=1):
    try:
        extractor = SequenceExtractor(**extractor_params)
        extractor.fit(sample_df)
        out = extractor.transform(sample_df)
        n_channels = out["X"].shape[2]
        key_params = {k: extractor_params[k] for k in ["acc_modes", "rotation_modes", "maxlen", "sampling_rate"] if k in extractor_params}
        print(f"Preview {i}: {key_params} -> channels={n_channels}, timesteps={out['X'].shape[1]}")
    except Exception as exc:
        print(f"Preview {i} failed: {exc}")

Bayesian mode: sampling 3 extractor configs for channel preview

Preview 1: {'acc_modes': 'smoothed|velocity|jerk', 'rotation_modes': 'quaternion|angular_velocity'} -> channels=129, timesteps=350
Preview 2: {'acc_modes': 'smoothed|velocity|jerk', 'rotation_modes': 'quaternion|angular_velocity'} -> channels=129, timesteps=50
Preview 3: {'acc_modes': 'smoothed|velocity|jerk', 'rotation_modes': 'quaternion|angular_velocity'} -> channels=129, timesteps=300


In [7]:
# ============================================================
# PIPELINE & SEARCH
# ============================================================
pipe = Pipeline([
    ("extractor", SequenceExtractor(padding_value=0.0)),
    (
        "classifier",
        KerasContrastiveSiameseClassifier(
            target=target_col,
            epochs=default_epochs,
            patience=default_patience,
            batch_size=default_batch_size,
            verbose=0,
            random_state=random_state,
        ),
    ),
])

if search_mode == "bayesian" and SKOPT_AVAILABLE:
    search = BayesSearchCV(
        pipe,
        param_space,
        n_iter=n_iter,
        scoring=scorer,
        cv=cv_object,
        n_jobs=1,
        random_state=random_state,
        error_score=error_score_constant,
        verbose=verbose,
        return_train_score=True,
    )
else:
    search = GridSearchCV(
        pipe,
        param_space,
        scoring=scorer,
        cv=cv_object,
        n_jobs=1,
        error_score=error_score_constant,
        verbose=verbose,
        return_train_score=True,
    )

print(f"Running {search_mode} search...")
search.fit(X_train, y_train, groups=groups)

print(f"\nBest CV score: {search.best_score_:.4f}")
print("Best parameters:")
print(search.best_params_)

best_model = search.best_estimator_

Running bayesian search...
Fitting 1 folds for each of 1 candidates, totalling 1 fits


I0000 00:00:1782696976.347045      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782696976.352823      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1782696982.417008      69 service.cc:152] XLA service 0x7e14fc2c7100 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782696982.417049      69 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1782696982.417053      69 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1782696983.408791      69 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1782696989.344121      69 device_compiler.h:188] Compiled clust

[CV 1/1] END classifier__backbone_filters=128-64, classifier__batch_size=256, classifier__contrastive_weight=0.9335393188593556, classifier__dense_units=256-128, classifier__dropout=0.0, classifier__embedding_dim=256, classifier__epochs=100, classifier__kernel_sizes=5-5-5-5, classifier__learning_rate=0.0001, classifier__padding_value=0.0, classifier__patience=1000, classifier__random_state=42, classifier__target=bfrb, classifier__temperature=0.5872812839278276, classifier__temporal_mode=attention, classifier__verbose=0, extractor__acc_modes=smoothed|velocity|jerk, extractor__chunk_stride=300, extractor__chunk_window_size=10, extractor__clip_value=None, extractor__dead_reckoning_detrend=False, extractor__interp_mode=linear, extractor__kalman_measurement_noise=0.01, extractor__kalman_process_noise=0.01, extractor__motion_filter_mode=None, extractor__native_sampling_rate=100, extractor__padding_value=0.0, extractor__rotation_modes=quaternion|angular_velocity, extractor__target_sampling_ra

In [8]:
# Create a timestamp for the filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# --- 1. Save the full cv_results_ DataFrame ---
cv_results_df = pd.DataFrame(search.cv_results_)
cv_results_df.to_csv(f"siamese_cv_results_{timestamp}.csv", index=False)
print(f"✅ Full CV results saved to siamese_cv_results_{timestamp}.csv")

# --- 2. Save a summary of the best parameters and scores ---
best_summary = {
    "best_score": search.best_score_,
    "best_params": str(search.best_params_),
    "mean_fit_time": search.cv_results_['mean_fit_time'][search.best_index_],
    "std_fit_time": search.cv_results_['std_fit_time'][search.best_index_],
    "mean_test_score": search.cv_results_['mean_test_score'][search.best_index_],
    "std_test_score": search.cv_results_['std_test_score'][search.best_index_],
}
best_df = pd.DataFrame([best_summary])
best_df.to_csv(f"siamese_best_summary_{timestamp}.csv", index=False)
print(f"✅ Best summary saved to siamese_best_summary_{timestamp}.csv")

# --- 3. (Optional) Save the best parameters as a readable text file ---
with open(f"siamese_best_params_{timestamp}.txt", "w") as f:
    f.write(f"Best CV Score: {search.best_score_:.4f}\n")
    f.write("Best Parameters:\n")
    for key, value in search.best_params_.items():
        f.write(f"  {key}: {value}\n")
print(f"✅ Best parameters saved to siamese_best_params_{timestamp}.txt")

print("\n✅ All CV results saved successfully.")

✅ Full CV results saved to siamese_cv_results_20260629_030611.csv
✅ Best summary saved to siamese_best_summary_20260629_030611.csv
✅ Best parameters saved to siamese_best_params_20260629_030611.txt

✅ All CV results saved successfully.


In [9]:
# ============================================================
# HOLDOUT EVALUATION
# ============================================================
if not do_cross_val and len(X_test) > 0:
    y_pred = best_model.predict(X_test)
    y_test_seq = y_test.drop_duplicates("sequence_id").sort_values("sequence_id")

    if target_col == "bfrb":
        eval_dict = evaluate_holdout(y_test_seq, y_pred, target_col=target_col, verbose=True)
        print(f"Holdout Competition Score: {eval_dict['competition_score']:.4f}")
    else:
        print("\nClassification Report:")
        print(classification_report(y_test_seq[target_col], y_pred, zero_division=0))
        macro_f1 = f1_score(y_test_seq[target_col], y_pred, average="macro", zero_division=0)
        print(f"Macro F1 Score: {macro_f1:.4f}")
else:
    print("Holdout evaluation skipped (cross-val mode or empty test set).")


FINAL EVALUATION
Binary F1 (non_bfrb vs bfrb): 0.9506
BFRB Gesture Macro F1: 0.3973
COMPETITION SCORE: 0.6740

----------------------------------------
BFRB Gesture Classification Report
----------------------------------------
                          precision    recall  f1-score   support

   Above ear - pull hair       0.68      0.52      0.59        81
      Cheek - pinch skin       0.34      0.38      0.36        81
     Eyebrow - pull hair       0.34      0.31      0.32        81
     Eyelash - pull hair       0.36      0.42      0.39        81
Forehead - pull hairline       0.53      0.49      0.51        81
      Forehead - scratch       0.71      0.49      0.58        81
       Neck - pinch skin       0.39      0.44      0.41        81
          Neck - scratch       0.52      0.33      0.41        81
                non_bfrb       0.00      0.00      0.00         0

                accuracy                           0.42       648
               macro avg       0.43      0.